# LFW 01. ArcFace embedding extraction

목표: manifest의 얼굴 이미지에서 정규화된 ArcFace 512D 임베딩을 추출해 PostgreSQL 원본 임베딩 테이블에 저장합니다. 성공 기준은 처리/성공/실패/건너뜀 수가 phase log에 남고 run별 중복 삽입이 없는 것입니다.

> **재시작/재개 규칙(필수)**: 임의 셀에서 시작하지 말고 **Kernel Restart 후 Run All**을 사용합니다. 00의 config hash와 `RUN_DIR/run_manifest.json`이 일치할 때만 재개하십시오. 중단되면 01 위에서부터 다시 실행하며, 같은 run의 이미 저장된 행은 건너뜁니다. 00 입력이 바뀌었으면 00부터 새 run을 만듭니다. `COMPLETED` run은 재개하지 않습니다.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate.resolve()
    raise RuntimeError('Run Jupyter from the ronbun repository or one of its subdirectories.')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
from research.runtime import RunStore, resolve_active_run

EXECUTE_STAGE = True
RUN_ROOT = PROJECT_ROOT / 'runs' / 'lfw'
LEGACY_RUN_ROOT = PROJECT_ROOT / 'runs'
try:
    RUN_DIR = resolve_active_run(RUN_ROOT)
except FileNotFoundError:
    RUN_ROOT = LEGACY_RUN_ROOT
    RUN_DIR = resolve_active_run(RUN_ROOT)
LIMIT = None  # Set a small integer for a recorded smoke run; use None for the full manifest.
USE_CUDA = os.environ.get('RONBUN_USE_CUDA', '0') == '1'


## Plan

- Attach to the frozen run and reject completed/mismatched state.
- Decode each image, select the largest detected face deterministically, and verify 512D L2 normalization.
- Store the original vector with run/model metadata; record failures without hiding them.


In [ ]:
def attach_run(run_dir: Path) -> tuple[RunStore, dict]:
    run = RunStore.open(run_dir)
    manifest = json.loads(run.manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') == 'completed' or (run_dir / 'COMPLETED').exists():
        raise RuntimeError('Completed runs are immutable; start again from notebook 00.')
    return run, manifest

preflight = {
    'execute_stage': EXECUTE_STAGE,
    'run_dir_resolved': str(RUN_DIR),
    'run_manifest_exists': bool(RUN_DIR and (RUN_DIR / 'run_manifest.json').is_file()),
    'cuda_requested': USE_CUDA,
    'limit': LIMIT,
}
preflight


## Execute and record

DB 비밀번호는 노트북에 쓰지 않습니다. `RONBUN_DB_PASSWORD` 또는 git에서 제외된 `configs/database.local.yaml`을 사용합니다.


In [ ]:
result = {'status': 'not_executed', **preflight}
if EXECUTE_STAGE:
    import cv2
    import pandas as pd
    from research.compression import ORIGIN_512
    from research.database import VectorRepository, create_database_engine, init_database, load_database_settings, session_scope
    from research.embeddings import ArcFaceFeatureExtractor, FaceAnalysisSettings
    from research.runtime.hashing import sha256_file
    from research.runtime.redaction import redact

    run, run_manifest = attach_run(RUN_DIR)
    run.verify_inputs()
    config = run_manifest['config']
    manifest_path = PROJECT_ROOT / config['dataset']['manifest_path']
    rows = pd.read_csv(manifest_path)
    if LIMIT is not None:
        rows = rows.head(int(LIMIT))
    providers = ('CUDAExecutionProvider', 'CPUExecutionProvider') if USE_CUDA else ('CPUExecutionProvider',)
    extractor = ArcFaceFeatureExtractor(FaceAnalysisSettings(providers=providers))
    engine = create_database_engine(load_database_settings())
    init_database(engine)
    counts = {'processed': 0, 'inserted': 0, 'skipped': 0, 'failed': 0}
    failures = []
    ledger = []
    with run.phase('01_arcface_embedding_extraction') as phase:
        with session_scope(engine) as session:
            repository = VectorRepository(session)
            for row in rows.to_dict(orient='records'):
                counts['processed'] += 1
                image_path = Path(str(row['image_path']))
                image_path = image_path if image_path.is_absolute() else PROJECT_ROOT / image_path
                content_sha256 = None
                try:
                    if not image_path.is_file():
                        raise FileNotFoundError(image_path)
                    content_sha256 = sha256_file(image_path)
                    file_size_bytes = image_path.stat().st_size
                    with session.begin_nested():
                        image = repository.add_image(
                            str(image_path), label=str(row['identity_id']),
                            content_sha256=content_sha256, file_size_bytes=file_size_bytes,
                        )
                        existing = repository.get_embeddings_512(
                            image_id=image.id, vector_type=ORIGIN_512, run_uid=run.run_id
                        )
                        if existing:
                            counts['skipped'] += 1
                            ledger.append({'image_path': str(image_path), 'content_sha256': content_sha256,
                                           'status': 'skipped_existing', 'embedding_id': existing[0].id})
                            continue
                        pixels = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
                        extracted = extractor.extract_with_metadata(pixels)
                        embedding_record = repository.add_embedding_512(
                            image.id, ORIGIN_512,
                            {'run_id': run.run_id, 'model': extractor.settings.model_name, 'l2_normalized': True},
                            extracted.embedding,
                            log=json.dumps({'face_count': extracted.face_count, 'bbox': extracted.bbox, 'detection_score': extracted.detection_score}),
                            run_uid=run.run_id,
                        )
                        counts['inserted'] += 1
                        ledger.append({'image_path': str(image_path), 'content_sha256': content_sha256,
                                       'status': 'inserted', 'embedding_id': embedding_record.id})
                except Exception as exc:
                    counts['failed'] += 1
                    failure = redact({'image_path': str(image_path), 'content_sha256': content_sha256,
                                      'error_type': type(exc).__name__, 'message': str(exc)})
                    failures.append(failure)
                    ledger.append(redact({'image_path': str(image_path), 'content_sha256': content_sha256,
                                          'status': 'failed', 'embedding_id': None,
                                          'error_type': type(exc).__name__, 'message': str(exc)}))
        report_path = phase.attempt_dir / f'extraction_summary_A{phase.attempt:03d}.json'
        ledger_path = phase.attempt_dir / f'extraction_ledger_A{phase.attempt:03d}.csv'
        report_path.write_text(json.dumps(redact({'counts': counts, 'failures': failures}), ensure_ascii=False, indent=2), encoding='utf-8')
        pd.DataFrame.from_records(redact(ledger)).to_csv(ledger_path, index=False)
        phase.publish_artifact(report_path)
        phase.publish_artifact(ledger_path)
        phase.record_counts(**counts)
    result = {'status': 'completed', 'run_id': run.run_id, **counts}
result


## Next step

`failed`가 0인지, 누락이 의도된 것인지 확인한 뒤 02로 이동합니다. 실패 원인이 데이터/모델 설정 변경이라면 01부터 새 attempt로 재실행합니다.
